### Step 1 - Verifying the Databricks Environment

In [0]:
print("Spark Version :",spark.version)
print("Current Catalog : ",spark.catalog.currentCatalog())
print("Current Schema : ",spark.catalog.currentDatabase())

Spark Version : 4.1.0
Current Catalog :  workspace
Current Schema :  default


In [0]:
%sql

DROP SCHEMA IF EXISTS gold CASCADE;
DROP SCHEMA IF EXISTS silver CASCADE;
DROP SCHEMA IF EXISTS bronze CASCADE;


In [0]:
%sql
SHOW SCHEMAS

databaseName
cts-az-schema
default
information_schema


In [0]:
schemas_to_drop=["gold","silver"]

for schema in schemas_to_drop:
  spark.sql(f"DROP SCHEMA IF EXISTS `{schema}` CASCADE")
  print(f"Dropped Schema :{schema} dropped")

Dropped Schema :gold dropped
Dropped Schema :silver dropped


In [0]:
display(spark.sql("SHOW SCHEMAS"))

databaseName
cts-az-schema
default
information_schema


### STEP 2 -Import libraries

In [0]:
from __future__ import annotations

import io
import os
import requests
import pandas as pd
import pyarrow.parquet as pq

from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

### Step- 3 Create Medallion Schemas

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS bronze
COMMENT 'Raw source-aligned NYC taxi data';

CREATE SCHEMA IF NOT EXISTS silver
COMMENT 'Cleaned,validated and enriched NYC taxi data';

CREATE SCHEMA IF NOT EXISTS gold
COMMENT 'Business-ready  NYC taxi  analytical data';

SHOW SCHEMAS;


databaseName
bronze
cts-az-schema
default
gold
information_schema
silver


### Step-4 Define source URLs and sample size

In [0]:
TRIP_URL =(
     "https://d37ci6vzurychx.cloudfront.net/"
    "trip-data/yellow_tripdata_2024-01.parquet"
)

ZONE_URL = (
    "https://d37ci6vzurychx.cloudfront.net/"
    "misc/taxi_zone_lookup.csv"
)
TEACHING_SAMPLE_ROWS=100_000
LOCAL_TRIP_FILE="/tmp/yellow_tripdata_2024-01.parquet"

print("Trip URL :   ",TRIP_URL)
print("Zone URL :   ",ZONE_URL)
print("TEaching Sample Rows :",TEACHING_SAMPLE_ROWS)


Trip URL :    https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet
Zone URL :    https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
TEaching Sample Rows : 100000


### Step 5 - Download and read the taxi zone lookup

In [0]:
zone_response= requests.get(ZONE_URL, timeout=120)
zone_response.raise_for_status()

zones_pdf=pd.read_csv(io.BytesIO(zone_response.content))
zones_df=spark.createDataFrame(zones_pdf)

zones_df.printSchema()
print("Zone rows: ",zones_df.count())
display(zones_df.limit(20))

root
 |-- LocationID: long (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)

Zone rows:  265


LocationID,Borough,Zone,service_zone
1,EWR,Newark Airport,EWR
2,Queens,Jamaica Bay,Boro Zone
3,Bronx,Allerton/Pelham Gardens,Boro Zone
4,Manhattan,Alphabet City,Yellow Zone
5,Staten Island,Arden Heights,Boro Zone
6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone
7,Queens,Astoria,Boro Zone
8,Queens,Astoria Park,Boro Zone
9,Queens,Auburndale,Boro Zone
10,Queens,Baisley Park,Boro Zone


### Step 6- Download a controlled trip sample

In [0]:
if not os.path.exists(LOCAL_TRIP_FILE):
    with requests.get(TRIP_URL, stream=True, timeout=300) as response:
        response.raise_for_status()
        with open(LOCAL_TRIP_FILE, "wb") as output_file:
            for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                if chunk:
                    output_file.write(chunk)

parquet_file = pq.ParquetFile(LOCAL_TRIP_FILE)

batches = []
remaining = TEACHING_SAMPLE_ROWS

for batch in parquet_file.iter_batches(batch_size=min(50_000, TEACHING_SAMPLE_ROWS)):
    if remaining <= 0:
        break

    if batch.num_rows > remaining:
        batch = batch.slice(0, remaining)

    batches.append(batch)
    remaining -= batch.num_rows

    if remaining <= 0:
        break

if not batches:
    raise RuntimeError("The Parquet file did not contain any readable rows.")

sample_table = (
    batches[0].to_table()
    if len(batches) == 1
    else __import__("pyarrow").Table.from_batches(batches)
)

trips_pdf = sample_table.to_pandas()
trips_df = spark.createDataFrame(trips_pdf)

print("Loaded trip rows:", trips_df.count())
trips_df.printSchema()
display(trips_df.limit(20))

Loaded trip rows: 100000
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
2,2024-01-01T00:57:55.000Z,2024-01-01T01:17:43.000Z,1,1.72,1,N,186,79,2,17.7,1.0,0.5,0.0,0.0,1.0,22.7,2.5,0.0
1,2024-01-01T00:03:00.000Z,2024-01-01T00:09:36.000Z,1,1.8,1,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
1,2024-01-01T00:17:06.000Z,2024-01-01T00:35:01.000Z,1,4.7,1,N,236,79,1,23.3,3.5,0.5,3.0,0.0,1.0,31.3,2.5,0.0
1,2024-01-01T00:36:38.000Z,2024-01-01T00:44:56.000Z,1,1.4,1,N,79,211,1,10.0,3.5,0.5,2.0,0.0,1.0,17.0,2.5,0.0
1,2024-01-01T00:46:51.000Z,2024-01-01T00:52:57.000Z,1,0.8,1,N,211,148,1,7.9,3.5,0.5,3.2,0.0,1.0,16.1,2.5,0.0
1,2024-01-01T00:54:08.000Z,2024-01-01T01:26:31.000Z,1,4.7,1,N,148,141,1,29.6,3.5,0.5,6.9,0.0,1.0,41.5,2.5,0.0
2,2024-01-01T00:49:44.000Z,2024-01-01T01:15:47.000Z,2,10.82,1,N,138,181,1,45.7,6.0,0.5,10.0,0.0,1.0,64.95,0.0,1.75
1,2024-01-01T00:30:40.000Z,2024-01-01T00:58:40.000Z,0,3.0,1,N,246,231,2,25.4,3.5,0.5,0.0,0.0,1.0,30.4,2.5,0.0
2,2024-01-01T00:26:01.000Z,2024-01-01T00:54:12.000Z,1,5.44,1,N,161,261,2,31.0,1.0,0.5,0.0,0.0,1.0,36.0,2.5,0.0
2,2024-01-01T00:28:08.000Z,2024-01-01T00:29:16.000Z,1,0.04,1,N,113,113,2,3.0,1.0,0.5,0.0,0.0,1.0,8.0,2.5,0.0


### step 7 -Basic Source actions
Actions trigger Spark execution.

- count()
- first()
- take()
- show()
- display()
- writing a table

In [0]:
print("TRip Count: ",trips_df.count())
print("First Row: ",trips_df.first())
print("First 2  Rows: ",trips_df.take(2))
trips_df.show(5,truncate=False)


TRip Count:  100000
First Row:  Row(VendorID=2, tpep_pickup_datetime=datetime.datetime(2024, 1, 1, 0, 57, 55), tpep_dropoff_datetime=datetime.datetime(2024, 1, 1, 1, 17, 43), passenger_count=1, trip_distance=1.72, RatecodeID=1, store_and_fwd_flag='N', PULocationID=186, DOLocationID=79, payment_type=2, fare_amount=17.7, extra=1.0, mta_tax=0.5, tip_amount=0.0, tolls_amount=0.0, improvement_surcharge=1.0, total_amount=22.7, congestion_surcharge=2.5, Airport_fee=0.0)
First 2  Rows:  [Row(VendorID=2, tpep_pickup_datetime=datetime.datetime(2024, 1, 1, 0, 57, 55), tpep_dropoff_datetime=datetime.datetime(2024, 1, 1, 1, 17, 43), passenger_count=1, trip_distance=1.72, RatecodeID=1, store_and_fwd_flag='N', PULocationID=186, DOLocationID=79, payment_type=2, fare_amount=17.7, extra=1.0, mta_tax=0.5, tip_amount=0.0, tolls_amount=0.0, improvement_surcharge=1.0, total_amount=22.7, congestion_surcharge=2.5, Airport_fee=0.0), Row(VendorID=1, tpep_pickup_datetime=datetime.datetime(2024, 1, 1, 0, 3), tpep

# Bronze Layer
### Step -8 prepare  Bronze trip data

In [0]:
bronze_trips_df=(
    trips_df
    .withColumn("source_file",F.lit("Yello_tripdata_2024_01.parquet"))
    .withColumn("source_url",F.lit(TRIP_URL))
    .withColumn("source_system",F.lit("NYC_TLC"))
    .withColumn("ingestion_timestamp",F.current_timestamp())
    .withColumn("ingestion_date",F.current_date())
)

display(bronze_trips_df.limit(10))
    

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,source_file,source_url,source_system,ingestion_timestamp,ingestion_date
2,2024-01-01T00:57:55.000Z,2024-01-01T01:17:43.000Z,1,1.72,1,N,186,79,2,17.7,1.0,0.5,0.0,0.0,1.0,22.7,2.5,0.0,Yello_tripdata_2024_01.parquet,https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet,NYC_TLC,2026-07-23T15:18:14.549Z,2026-07-23
1,2024-01-01T00:03:00.000Z,2024-01-01T00:09:36.000Z,1,1.8,1,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0,Yello_tripdata_2024_01.parquet,https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet,NYC_TLC,2026-07-23T15:18:14.549Z,2026-07-23
1,2024-01-01T00:17:06.000Z,2024-01-01T00:35:01.000Z,1,4.7,1,N,236,79,1,23.3,3.5,0.5,3.0,0.0,1.0,31.3,2.5,0.0,Yello_tripdata_2024_01.parquet,https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet,NYC_TLC,2026-07-23T15:18:14.549Z,2026-07-23
1,2024-01-01T00:36:38.000Z,2024-01-01T00:44:56.000Z,1,1.4,1,N,79,211,1,10.0,3.5,0.5,2.0,0.0,1.0,17.0,2.5,0.0,Yello_tripdata_2024_01.parquet,https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet,NYC_TLC,2026-07-23T15:18:14.549Z,2026-07-23
1,2024-01-01T00:46:51.000Z,2024-01-01T00:52:57.000Z,1,0.8,1,N,211,148,1,7.9,3.5,0.5,3.2,0.0,1.0,16.1,2.5,0.0,Yello_tripdata_2024_01.parquet,https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet,NYC_TLC,2026-07-23T15:18:14.549Z,2026-07-23
1,2024-01-01T00:54:08.000Z,2024-01-01T01:26:31.000Z,1,4.7,1,N,148,141,1,29.6,3.5,0.5,6.9,0.0,1.0,41.5,2.5,0.0,Yello_tripdata_2024_01.parquet,https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet,NYC_TLC,2026-07-23T15:18:14.549Z,2026-07-23
2,2024-01-01T00:49:44.000Z,2024-01-01T01:15:47.000Z,2,10.82,1,N,138,181,1,45.7,6.0,0.5,10.0,0.0,1.0,64.95,0.0,1.75,Yello_tripdata_2024_01.parquet,https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet,NYC_TLC,2026-07-23T15:18:14.549Z,2026-07-23
1,2024-01-01T00:30:40.000Z,2024-01-01T00:58:40.000Z,0,3.0,1,N,246,231,2,25.4,3.5,0.5,0.0,0.0,1.0,30.4,2.5,0.0,Yello_tripdata_2024_01.parquet,https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet,NYC_TLC,2026-07-23T15:18:14.549Z,2026-07-23
2,2024-01-01T00:26:01.000Z,2024-01-01T00:54:12.000Z,1,5.44,1,N,161,261,2,31.0,1.0,0.5,0.0,0.0,1.0,36.0,2.5,0.0,Yello_tripdata_2024_01.parquet,https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet,NYC_TLC,2026-07-23T15:18:14.549Z,2026-07-23
2,2024-01-01T00:28:08.000Z,2024-01-01T00:29:16.000Z,1,0.04,1,N,113,113,2,3.0,1.0,0.5,0.0,0.0,1.0,8.0,2.5,0.0,Yello_tripdata_2024_01.parquet,https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet,NYC_TLC,2026-07-23T15:18:14.549Z,2026-07-23


### step 9 :Prepare Bronze Zone Data

In [0]:
bronze_zones_df=(
    zones_df
    .withColumnRenamed("LocationID","location_id")
    .withColumnRenamed("Borough","borough")
    .withColumnRenamed("Zone","zone_name")
    .withColumn("source_file",F.lit("taxi_zone_lookup.csv"))
    .withColumn("source_url",F.lit(ZONE_URL))
    .withColumn("ingestion_timestamp",F.current_timestamp())  
)

display(bronze_zones_df.limit(10))

location_id,borough,zone_name,service_zone,source_file,source_url,ingestion_timestamp
1,EWR,Newark Airport,EWR,taxi_zone_lookup.csv,https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv,2026-07-23T15:18:16.238Z
2,Queens,Jamaica Bay,Boro Zone,taxi_zone_lookup.csv,https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv,2026-07-23T15:18:16.238Z
3,Bronx,Allerton/Pelham Gardens,Boro Zone,taxi_zone_lookup.csv,https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv,2026-07-23T15:18:16.238Z
4,Manhattan,Alphabet City,Yellow Zone,taxi_zone_lookup.csv,https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv,2026-07-23T15:18:16.238Z
5,Staten Island,Arden Heights,Boro Zone,taxi_zone_lookup.csv,https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv,2026-07-23T15:18:16.238Z
6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone,taxi_zone_lookup.csv,https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv,2026-07-23T15:18:16.238Z
7,Queens,Astoria,Boro Zone,taxi_zone_lookup.csv,https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv,2026-07-23T15:18:16.238Z
8,Queens,Astoria Park,Boro Zone,taxi_zone_lookup.csv,https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv,2026-07-23T15:18:16.238Z
9,Queens,Auburndale,Boro Zone,taxi_zone_lookup.csv,https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv,2026-07-23T15:18:16.238Z
10,Queens,Baisley Park,Boro Zone,taxi_zone_lookup.csv,https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv,2026-07-23T15:18:16.238Z


### Step 10 -Write Bronze Delta Tables

In [0]:
(
    bronze_trips_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema","true")
    .saveAsTable("bronze.nyc_yellow_taxi_trips_raw")
)
(
    bronze_zones_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema","true")
    .saveAsTable("bronze.nyc_taxi_zones_raw")
)

print("Bronze table written successfully")

Bronze table written successfully


In [0]:
%sql
SHOW TABLES IN bronze;

database,tableName,isTemporary
bronze,nyc_taxi_zones_raw,false
bronze,nyc_yellow_taxi_trips_raw,false


In [0]:
%sql
SELECT COUNT(*) AS bronze_trip_count FROM bronze.nyc_yellow_taxi_trips_raw

bronze_trip_count
100000


### Step 11 - Common bronze Transformations

In [0]:
bronze_df = spark.table('bronze.nyc_yellow_taxi_trips_raw')
projection_df =bronze_df.select(
    F.col("VendorID").alias("vendor_id"),
    F.col("tpep_pickup_datetime").alias("pickup_datetime"),
    F.col("tpep_dropoff_datetime").alias("dropoff_datetime") ,  
    F.col("trip_distance"),
    F.col("fare_amount"),
  F.col("total_amount")
)
projection_df.show(10,truncate=False)

+---------+-------------------+-------------------+-------------+-----------+------------+
|vendor_id|pickup_datetime    |dropoff_datetime   |trip_distance|fare_amount|total_amount|
+---------+-------------------+-------------------+-------------+-----------+------------+
|2        |2024-01-01 16:46:03|2024-01-01 16:49:40|0.75         |5.8        |11.76       |
|2        |2024-01-01 16:53:03|2024-01-01 16:59:21|0.8          |7.9        |14.9        |
|2        |2024-01-01 16:06:49|2024-01-01 17:01:14|16.89        |70.0       |97.13       |
|2        |2024-01-01 16:31:42|2024-01-01 16:49:27|7.26         |30.3       |45.91       |
|2        |2024-01-01 16:27:50|2024-01-01 16:35:50|1.44         |10.0       |14.0        |
|2        |2024-01-01 16:52:32|2024-01-01 17:17:49|2.52         |22.6       |30.59       |
|2        |2024-01-01 16:08:34|2024-01-01 16:12:10|1.09         |6.5        |12.6        |
|2        |2024-01-01 16:17:59|2024-01-01 16:39:50|4.31         |24.0       |33.6        |

In [0]:
# Filter,whre, between ,isin and null checks.

filtered_df=bronze_df.filter(
    (F.col("trip_distance")>0)
    &(F.col("total_amount")>0)
    & F.col("payment_type").isin(1,2)
)

print("Filtered count : ",filtered_df.count())

#bronze_df.where(F.col("trip_distance").between(1,10)).show()
#bronze_df.filter(F.col("passenger_count").isNull()).show()
bronze_df.filter(F.col("passenger_count").isNotNull()).show()


Filtered count :  95315
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+--------------------+--------------------+-------------+--------------------+--------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|         source_file|          source_url|source_system| ingestion_timestamp|ingestion_date|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+-------------

### Step 12 -- Built in functions examples

In [0]:
# Null and conditional functions
bronze_df.select(
    "passenger_count",
    "RatecodeID",
    F.coalesce("passenger_count",F.lit(0)).alias("passenger_count_clean"),
    F.coalesce("RatecodeID",F.lit(-1)).alias("rate_code_clean"),
    F.when(F.col("passenger_count").isNull(),"MISSING")
    .otherwise("AVAILABLE")
    .alias("passenger_data_status")
).show(10)

+---------------+----------+---------------------+---------------+---------------------+
|passenger_count|RatecodeID|passenger_count_clean|rate_code_clean|passenger_data_status|
+---------------+----------+---------------------+---------------+---------------------+
|              5|         1|                    5|              1|            AVAILABLE|
|              2|         1|                    2|              1|            AVAILABLE|
|              1|         2|                    1|              2|            AVAILABLE|
|              1|         1|                    1|              1|            AVAILABLE|
|              1|         1|                    1|              1|            AVAILABLE|
|              1|         1|                    1|              1|            AVAILABLE|
|              1|         1|                    1|              1|            AVAILABLE|
|              2|         1|                    2|              1|            AVAILABLE|
|              1|    

In [0]:
%sql
SELECT * FROM bronze.nyc_taxi_zones_raw;

location_id,borough,zone_name,service_zone,source_file,source_url,ingestion_timestamp
1,EWR,Newark Airport,EWR,taxi_zone_lookup.csv,https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv,2026-07-23T15:18:34.285Z
2,Queens,Jamaica Bay,Boro Zone,taxi_zone_lookup.csv,https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv,2026-07-23T15:18:34.285Z
3,Bronx,Allerton/Pelham Gardens,Boro Zone,taxi_zone_lookup.csv,https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv,2026-07-23T15:18:34.285Z
4,Manhattan,Alphabet City,Yellow Zone,taxi_zone_lookup.csv,https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv,2026-07-23T15:18:34.285Z
5,Staten Island,Arden Heights,Boro Zone,taxi_zone_lookup.csv,https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv,2026-07-23T15:18:34.285Z
6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone,taxi_zone_lookup.csv,https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv,2026-07-23T15:18:34.285Z
7,Queens,Astoria,Boro Zone,taxi_zone_lookup.csv,https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv,2026-07-23T15:18:34.285Z
8,Queens,Astoria Park,Boro Zone,taxi_zone_lookup.csv,https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv,2026-07-23T15:18:34.285Z
9,Queens,Auburndale,Boro Zone,taxi_zone_lookup.csv,https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv,2026-07-23T15:18:34.285Z
10,Queens,Baisley Park,Boro Zone,taxi_zone_lookup.csv,https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv,2026-07-23T15:18:34.285Z


In [0]:
# String and regular-expression functions using zone data.
spark.table("bronze.nyc_taxi_zones_raw").select(
    "location_id",
    F.trim("borough").alias("borough_trimmed"),
    F.upper(F.trim("borough")).alias("borough_upper"),
    F.lower(F.trim("zone_name")).alias("zone_lower"),
    F.initcap(F.trim("zone_name")).alias("zone_title"),
    F.length(F.trim("zone_name")).alias("zone_name_length"),
    F.substring(F.trim("zone_name"), 1, 12).alias("zone_short"),
    F.concat_ws(" - ", "borough", "zone_name").alias("zone_label"),
    F.regexp_replace("zone_name", "[^A-Za-z0-9 ]", "").alias("zone_alphanumeric"),
    F.regexp_extract("zone_name", "([A-Za-z]+)", 1).alias("first_word"),
    F.split("zone_name", " ").alias("zone_words")
).show(10, truncate=False)

+-----------+---------------+-------------+-----------------------+-----------------------+----------------+------------+---------------------------------------+----------------------+----------+--------------------------+
|location_id|borough_trimmed|borough_upper|zone_lower             |zone_title             |zone_name_length|zone_short  |zone_label                             |zone_alphanumeric     |first_word|zone_words                |
+-----------+---------------+-------------+-----------------------+-----------------------+----------------+------------+---------------------------------------+----------------------+----------+--------------------------+
|1          |EWR            |EWR          |newark airport         |Newark Airport         |14              |Newark Airpo|EWR - Newark Airport                   |Newark Airport        |Newark    |[Newark, Airport]         |
|2          |Queens         |QUEENS       |jamaica bay            |Jamaica Bay            |11              |

In [0]:
# Date and timestamp functions.
bronze_df.select(
    F.col("tpep_pickup_datetime").alias("pickup_timestamp"),
    F.col("tpep_dropoff_datetime").alias("dropoff_timestamp"),
    F.to_date("tpep_pickup_datetime").alias("pickup_date"),
    F.year("tpep_pickup_datetime").alias("pickup_year"),
    F.month("tpep_pickup_datetime").alias("pickup_month"),
    F.dayofmonth("tpep_pickup_datetime").alias("pickup_day"),
    F.dayofweek("tpep_pickup_datetime").alias("pickup_day_of_week"),
    F.hour("tpep_pickup_datetime").alias("pickup_hour"),
    F.date_format("tpep_pickup_datetime", "EEEE").alias("pickup_day_name"),
    F.date_format("tpep_pickup_datetime", "yyyy-MM").alias("pickup_year_month"),
    F.round(
        (
            F.unix_timestamp("tpep_dropoff_datetime")
            - F.unix_timestamp("tpep_pickup_datetime")
        ) / 60.0,
        2
    ).alias("duration_minutes")
).show(10, truncate=False)

+-------------------+-------------------+-----------+-----------+------------+----------+------------------+-----------+---------------+-----------------+----------------+
|pickup_timestamp   |dropoff_timestamp  |pickup_date|pickup_year|pickup_month|pickup_day|pickup_day_of_week|pickup_hour|pickup_day_name|pickup_year_month|duration_minutes|
+-------------------+-------------------+-----------+-----------+------------+----------+------------------+-----------+---------------+-----------------+----------------+
|2024-01-01 16:46:03|2024-01-01 16:49:40|2024-01-01 |2024       |1           |1         |2                 |16         |Monday         |2024-01          |3.62            |
|2024-01-01 16:53:03|2024-01-01 16:59:21|2024-01-01 |2024       |1           |1         |2                 |16         |Monday         |2024-01          |6.3             |
|2024-01-01 16:06:49|2024-01-01 17:01:14|2024-01-01 |2024       |1           |1         |2                 |16         |Monday         |2024

In [0]:
# Complex types: array, struct, map, JSON.
complex_df = bronze_df.select(
    F.array("PULocationID", "DOLocationID").alias("route_location_ids"),
    F.struct(
        "PULocationID",
        "DOLocationID",
        "trip_distance"
    ).alias("route_struct"),
    F.create_map(
        F.lit("fare"), F.col("fare_amount"),
        F.lit("tip"), F.col("tip_amount"),
        F.lit("total"), F.col("total_amount")
    ).alias("amount_map")
)

complex_df.show(5, truncate=False)


+------------------+-----------------+--------------------------------------------+
|route_location_ids|route_struct     |amount_map                                  |
+------------------+-----------------+--------------------------------------------+
|[141, 229]        |{141, 229, 0.75} |{fare -> 5.8, tip -> 1.96, total -> 11.76}  |
|[100, 164]        |{100, 164, 0.8}  |{fare -> 7.9, tip -> 3.0, total -> 14.9}    |
|[164, 132]        |{164, 132, 16.89}|{fare -> 70.0, tip -> 16.19, total -> 97.13}|
|[138, 80]         |{138, 80, 7.26}  |{fare -> 30.3, tip -> 7.36, total -> 45.91} |
|[239, 142]        |{239, 142, 1.44} |{fare -> 10.0, tip -> 0.0, total -> 14.0}   |
+------------------+-----------------+--------------------------------------------+
only showing top 5 rows


In [0]:
complex_df.select(
    F.size("route_location_ids").alias("route_array_size"),
    F.element_at("route_location_ids", 1).alias("pickupte_location_id"),
   F.element_at("amount_map", "total").alias("mapped_total_amount"),
   ).show(5)

+----------------+--------------------+-------------------+
|route_array_size|pickupte_location_id|mapped_total_amount|
+----------------+--------------------+-------------------+
|               2|                 233|               15.4|
|               2|                 164|              97.13|
|               2|                 113|               51.2|
|               2|                 186|               19.6|
|               2|                 233|               13.3|
+----------------+--------------------+-------------------+
only showing top 5 rows


### Step - 18 JSON Functions

In [0]:
# JSON functions
json_source_df=bronze_df.select(
    F.to_json(F.struct(
        "VendorID",
        "PULocationID",
        "DOLocationID",
        "trip_distance",
        "total_amount"
    )).alias("trip_json")
)

json_df.show(5, truncate=False)


+-----------------------------------------------------------------------------------------------+
|trip_json                                                                                      |
+-----------------------------------------------------------------------------------------------+
|{"VendorID":2,"PULocationID":141,"DOLocationID":229,"trip_distance":0.75,"total_amount":11.76} |
|{"VendorID":2,"PULocationID":100,"DOLocationID":164,"trip_distance":0.8,"total_amount":14.9}   |
|{"VendorID":2,"PULocationID":164,"DOLocationID":132,"trip_distance":16.89,"total_amount":97.13}|
|{"VendorID":2,"PULocationID":138,"DOLocationID":80,"trip_distance":7.26,"total_amount":45.91}  |
|{"VendorID":2,"PULocationID":239,"DOLocationID":142,"trip_distance":1.44,"total_amount":14.0}  |
+-----------------------------------------------------------------------------------------------+
only showing top 5 rows


In [0]:
json_schema=T.StructType([
    T.StructField("VendorID", T.LongType(),True),
    T.StructField("PULocationID", T.LongType(),True),
    T.StructField("DOLocationID", T.LongType(),True),
    T.StructField("trip_distance", T.DoubleType(),True),
    T.StructField("total_amount", T.DoubleType(),True)
])

parsed_json_df=json_source_df.select(
    F.from_json("trip_json",json_schema).alias("trip")
)

parsed_json_df.select("trip.*").show(5,truncate=False)


+--------+------------+------------+-------------+------------+
|VendorID|PULocationID|DOLocationID|trip_distance|total_amount|
+--------+------------+------------+-------------+------------+
|1       |233         |236         |2.2          |15.4        |
|2       |164         |132         |16.22        |97.13       |
|1       |113         |138         |11.1         |51.2        |
|1       |186         |140         |2.9          |19.6        |
|1       |233         |164         |1.4          |13.3        |
+--------+------------+------------+-------------+------------+
only showing top 5 rows


# Build Silver Layer

### step 19- Define valid and invalid trip conditions

The Silver layer applies business rules.

A trip is considered valid when:

- Pickup timestamp is present.
- Drop-off timestamp is present.
- Drop-off occurs after pickup.
- Trip distance is greater than zero.
- Total amount is non-negative.
- Pickup and drop-off zone IDs are present.
- Trip duration is between 1 minute and 24 hours.

### step 20: clean and standarized trips

In [0]:
raw_trips_df = spark.table("bronze.nyc_yellow_taxi_trips_raw")

silver_prepared_df = (
    raw_trips_df
    .select(
        F.col("VendorID").cast("int").alias("vendor_id"),
        F.col("tpep_pickup_datetime").cast("timestamp").alias("pickup_timestamp"),
        F.col("tpep_dropoff_datetime").cast("timestamp").alias("dropoff_timestamp"),
        F.coalesce(F.col("passenger_count").cast("int"), F.lit(0)).alias("passenger_count"),
        F.coalesce(F.col("trip_distance").cast("double"), F.lit(0.0)).alias("trip_distance_miles"),
        F.col("RatecodeID").cast("int").alias("rate_code_id"),
        F.col("store_and_fwd_flag").cast("string").alias("store_and_forward_flag"),
        F.col("PULocationID").cast("int").alias("pickup_location_id"),
        F.col("DOLocationID").cast("int").alias("dropoff_location_id"),
        F.col("payment_type").cast("int").alias("payment_type_id"),
        F.coalesce(F.col("fare_amount").cast("decimal(12,2)"), F.lit(0)).alias("fare_amount"),
        F.coalesce(F.col("extra").cast("decimal(12,2)"), F.lit(0)).alias("extra_amount"),
        F.coalesce(F.col("mta_tax").cast("decimal(12,2)"), F.lit(0)).alias("mta_tax"),
        F.coalesce(F.col("tip_amount").cast("decimal(12,2)"), F.lit(0)).alias("tip_amount"),
        F.coalesce(F.col("tolls_amount").cast("decimal(12,2)"), F.lit(0)).alias("tolls_amount"),
        F.coalesce(F.col("improvement_surcharge").cast("decimal(12,2)"), F.lit(0)).alias("improvement_surcharge"),
        F.coalesce(F.col("total_amount").cast("decimal(12,2)"), F.lit(0)).alias("total_amount"),
        F.coalesce(F.col("congestion_surcharge").cast("decimal(12,2)"), F.lit(0)).alias("congestion_surcharge"),
        "source_file",
        "ingestion_timestamp"
    )
    .withColumn(
        "trip_duration_minutes",
        F.round(
            (
                F.unix_timestamp("dropoff_timestamp")
                - F.unix_timestamp("pickup_timestamp")
            ) / 60.0,
            2
        )
    )
    .withColumn("pickup_date", F.to_date("pickup_timestamp"))
    .withColumn("pickup_year", F.year("pickup_timestamp"))
    .withColumn("pickup_month", F.month("pickup_timestamp"))
    .withColumn("pickup_day", F.dayofmonth("pickup_timestamp"))
    .withColumn("pickup_hour", F.hour("pickup_timestamp"))
    .withColumn("pickup_day_name", F.date_format("pickup_timestamp", "EEEE"))
    .withColumn(
        "payment_method",
        F.when(F.col("payment_type_id") == 1, "CREDIT_CARD")
         .when(F.col("payment_type_id") == 2, "CASH")
         .when(F.col("payment_type_id") == 3, "NO_CHARGE")
         .when(F.col("payment_type_id") == 4, "DISPUTE")
         .when(F.col("payment_type_id") == 5, "UNKNOWN")
         .when(F.col("payment_type_id") == 6, "VOIDED")
         .otherwise("UNMAPPED")
    )
    .withColumn(
        "distance_band",
        F.when(F.col("trip_distance_miles") < 2, "SHORT")
         .when(F.col("trip_distance_miles") < 8, "MEDIUM")
         .otherwise("LONG")
    )
    .withColumn(
        "tip_percentage",
        F.when(
            F.col("fare_amount") > 0,
            F.round((F.col("tip_amount") / F.col("fare_amount")) * 100, 2)
        ).otherwise(F.lit(0.0))
    )
)

silver_prepared_df.printSchema()
display(silver_prepared_df.limit(20))

root
 |-- vendor_id: integer (nullable = true)
 |-- pickup_timestamp: timestamp (nullable = true)
 |-- dropoff_timestamp: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = false)
 |-- trip_distance_miles: double (nullable = false)
 |-- rate_code_id: integer (nullable = true)
 |-- store_and_forward_flag: string (nullable = true)
 |-- pickup_location_id: integer (nullable = true)
 |-- dropoff_location_id: integer (nullable = true)
 |-- payment_type_id: integer (nullable = true)
 |-- fare_amount: decimal(12,2) (nullable = false)
 |-- extra_amount: decimal(12,2) (nullable = false)
 |-- mta_tax: decimal(12,2) (nullable = false)
 |-- tip_amount: decimal(12,2) (nullable = false)
 |-- tolls_amount: decimal(12,2) (nullable = false)
 |-- improvement_surcharge: decimal(12,2) (nullable = false)
 |-- total_amount: decimal(12,2) (nullable = false)
 |-- congestion_surcharge: decimal(12,2) (nullable = false)
 |-- source_file: string (nullable = true)
 |-- ingestion_timestamp: times

vendor_id,pickup_timestamp,dropoff_timestamp,passenger_count,trip_distance_miles,rate_code_id,store_and_forward_flag,pickup_location_id,dropoff_location_id,payment_type_id,fare_amount,extra_amount,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,source_file,ingestion_timestamp,trip_duration_minutes,pickup_date,pickup_year,pickup_month,pickup_day,pickup_hour,pickup_day_name,payment_method,distance_band,tip_percentage
1,2024-01-01T07:14:05.000Z,2024-01-01T07:20:37.000Z,1,2.2,1,N,233,236,2,11.40,2.50,0.50,0.00,0.00,1.00,15.40,2.50,Yello_tripdata_2024_01.parquet,2026-07-23T15:18:21.935Z,6.53,2024-01-01,2024,1,1,7,Monday,CASH,MEDIUM,0.0
2,2024-01-01T07:39:53.000Z,2024-01-01T08:03:05.000Z,3,16.22,2,N,164,132,1,70.00,0.00,0.50,16.19,6.94,1.00,97.13,2.50,Yello_tripdata_2024_01.parquet,2026-07-23T15:18:21.935Z,23.2,2024-01-01,2024,1,1,7,Monday,CREDIT_CARD,LONG,23.13
1,2024-01-01T07:51:51.000Z,2024-01-01T08:11:15.000Z,1,11.1,1,N,113,138,2,42.20,7.50,0.50,0.00,0.00,1.00,51.20,2.50,Yello_tripdata_2024_01.parquet,2026-07-23T15:18:21.935Z,19.4,2024-01-01,2024,1,1,7,Monday,CASH,LONG,0.0
1,2024-01-01T07:00:44.000Z,2024-01-01T07:13:00.000Z,1,2.9,1,N,186,140,2,15.60,2.50,0.50,0.00,0.00,1.00,19.60,2.50,Yello_tripdata_2024_01.parquet,2026-07-23T15:18:21.935Z,12.27,2024-01-01,2024,1,1,7,Monday,CASH,MEDIUM,0.0
1,2024-01-01T07:21:23.000Z,2024-01-01T07:28:00.000Z,1,1.4,1,N,233,164,2,9.30,2.50,0.50,0.00,0.00,1.00,13.30,2.50,Yello_tripdata_2024_01.parquet,2026-07-23T15:18:21.935Z,6.62,2024-01-01,2024,1,1,7,Monday,CASH,SHORT,0.0
1,2024-01-01T07:33:57.000Z,2024-01-01T07:46:16.000Z,3,3.1,1,N,186,45,1,15.60,2.50,0.50,0.00,0.00,1.00,19.60,2.50,Yello_tripdata_2024_01.parquet,2026-07-23T15:18:21.935Z,12.32,2024-01-01,2024,1,1,7,Monday,CREDIT_CARD,MEDIUM,0.0
2,2024-01-01T07:14:06.000Z,2024-01-01T07:19:26.000Z,1,0.92,1,N,262,237,1,7.90,0.00,0.50,2.00,0.00,1.00,13.90,2.50,Yello_tripdata_2024_01.parquet,2026-07-23T15:18:21.935Z,5.33,2024-01-01,2024,1,1,7,Monday,CREDIT_CARD,SHORT,25.32
2,2024-01-01T07:21:27.000Z,2024-01-01T07:36:20.000Z,1,6.87,1,N,238,243,2,28.90,0.00,0.50,0.00,0.00,1.00,32.90,2.50,Yello_tripdata_2024_01.parquet,2026-07-23T15:18:21.935Z,14.88,2024-01-01,2024,1,1,7,Monday,CASH,MEDIUM,0.0
1,2024-01-01T07:06:36.000Z,2024-01-01T07:31:14.000Z,1,11.4,1,N,132,201,1,44.30,1.75,0.50,10.60,5.60,1.00,63.75,0.00,Yello_tripdata_2024_01.parquet,2026-07-23T15:18:21.935Z,24.63,2024-01-01,2024,1,1,7,Monday,CREDIT_CARD,LONG,23.93
2,2024-01-01T07:22:59.000Z,2024-01-01T07:46:32.000Z,1,3.23,5,N,48,249,4,-9.90,0.00,-0.50,0.00,0.00,-1.00,-13.90,-2.50,Yello_tripdata_2024_01.parquet,2026-07-23T15:18:21.935Z,23.55,2024-01-01,2024,1,1,7,Monday,DISPUTE,MEDIUM,0.0


### Step 21 -Add validation messages

In [0]:
validated_df=(
    silver_prepared_df
    .withColumn(
        "validation_message",
        F.concat_ws(
            "; ",
            F.when(F.col("pickup_timestamp").isNull(),"Missing pickup timestamp"),
            F.when(F.col("dropoff_timestamp").isNull(),"Missing dropo-ff timestamp"),
             F.when(F.col("dropoff_timestamp") <= F.col("pickup_timestamp"), "Drop-off is not after pickup"),
            F.when(F.col("trip_distance_miles") <= 0, "Trip distance must be greater than zero"),
            F.when(F.col("total_amount") < 0, "Total amount cannot be negative"),
            F.when(F.col("pickup_location_id").isNull(), "Missing pickup location"),
            F.when(F.col("dropoff_location_id").isNull(), "Missing drop-off location"),
            F.when(F.col("trip_duration_minutes") < 1, "Trip duration is below one minute"),
            F.when(F.col("trip_duration_minutes") > 1440, "Trip duration exceeds 24 hours")
        )
        )
    .withColumn(
        "record_status",
        F.when(F.length(F.col("validation_message")) == 0, "VALID")
         .otherwise("INVALID")
    )
)

validated_df.groupBy("record_status").count().show()
        

+-------------+-----+
|record_status|count|
+-------------+-----+
|        VALID|96326|
|      INVALID| 3674|
+-------------+-----+



### Step 22: Create deterministic trip identifier and remove duplicates

In [0]:
with_trip_id_df = validated_df.withColumn(
    "trip_id",
    F.sha2(
        F.concat_ws(
            "||",
            F.coalesce(F.col("vendor_id").cast("string"), F.lit("NULL")),
            F.coalesce(F.col("pickup_timestamp").cast("string"), F.lit("NULL")),
            F.coalesce(F.col("dropoff_timestamp").cast("string"), F.lit("NULL")),
            F.coalesce(F.col("pickup_location_id").cast("string"), F.lit("NULL")),
            F.coalesce(F.col("dropoff_location_id").cast("string"), F.lit("NULL")),
            F.coalesce(F.col("total_amount").cast("string"), F.lit("NULL"))
        ),
        256
    )
)

trip_dedup_window = Window.partitionBy("trip_id").orderBy(
    F.col("ingestion_timestamp").desc(),
    F.col("source_file").asc()
)

deduplicated_df = (
    with_trip_id_df
    .withColumn("duplicate_row_number", F.row_number().over(trip_dedup_window))
)

### write invalid rows

In [0]:
invalid_trips_df=deduplicated_df.filter(
    (F.col("record_status")=="INVALID")
    | (F.col("duplicate_row_number")>1)
)

(
    invalid_trips_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.invalid_taxi_trips")
)

### Write valid, deduplicated rows:

In [0]:
clean_trips_df = (
    deduplicated_df
    .filter(
        (F.col("record_status") == "VALID")
        & (F.col("duplicate_row_number") == 1)
    )
    .drop("duplicate_row_number", "record_status", "validation_message")
)

(
    clean_trips_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("pickup_date")
    .saveAsTable("silver.clean_taxi_trips")
)



### Step 23: Clean and standardize zone reference data


In [0]:
clean_zones_df = (
    spark.table("bronze.nyc_taxi_zones_raw")
    .select(
        F.col("location_id").cast("int").alias("location_id"),
        F.initcap(F.trim("borough")).alias("borough"),
        F.initcap(F.trim("zone_name")).alias("zone_name"),
        F.upper(F.trim("service_zone")).alias("service_zone"),
        "source_file",
        "ingestion_timestamp"
    )
    .filter(F.col("location_id").isNotNull())
    .dropDuplicates(["location_id"])
)

(
    clean_zones_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.clean_taxi_zones")
)

### Step 24: Join trips with pickup and drop-off zones

In [0]:
trips = spark.table("silver.clean_taxi_trips").alias("t")
pickup_zones = spark.table("silver.clean_taxi_zones").alias("pu")
dropoff_zones = spark.table("silver.clean_taxi_zones").alias("do")

enriched_trips_df = (
    trips
    .join(
        pickup_zones,
        F.col("t.pickup_location_id") == F.col("pu.location_id"),
        "left"
    )
    .join(
        dropoff_zones,
        F.col("t.dropoff_location_id") == F.col("do.location_id"),
        "left"
    )
    .select(
        "t.*",
        F.col("pu.borough").alias("pickup_borough"),
        F.col("pu.zone_name").alias("pickup_zone"),
        F.col("pu.service_zone").alias("pickup_service_zone"),
        F.col("do.borough").alias("dropoff_borough"),
        F.col("do.zone_name").alias("dropoff_zone"),
        F.col("do.service_zone").alias("dropoff_service_zone")
    )
    .withColumn(
        "route_name",
        F.concat_ws(" -> ", "pickup_zone", "dropoff_zone")
    )
)

(
    enriched_trips_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("pickup_date")
    .saveAsTable("silver.enriched_taxi_trips")
)

# Silver SQL implementation
### Step 25: Query the Silver table with SQL

In [0]:
%sql
SELECT
    pickup_date,
    pickup_borough,
    pickup_zone,
    dropoff_borough,
    dropoff_zone,
    payment_method,
    trip_distance_miles,
    trip_duration_minutes,
    fare_amount,
    tip_amount,
    total_amount
FROM silver.enriched_taxi_trips
ORDER BY pickup_timestamp
LIMIT 20

pickup_date,pickup_borough,pickup_zone,dropoff_borough,dropoff_zone,payment_method,trip_distance_miles,trip_duration_minutes,fare_amount,tip_amount,total_amount
2002-12-31,Manhattan,Murray Hill,Manhattan,Murray Hill,NO_CHARGE,0.63,6.03,6.50,0.00,10.50
2009-01-01,Manhattan,Kips Bay,Unknown,null,CASH,0.46,3.0,4.40,0.00,9.40
2023-12-31,Manhattan,Flatiron,Manhattan,East Chelsea,CREDIT_CARD,0.47,2.72,5.10,0.00,10.10
2023-12-31,Manhattan,West Chelsea/hudson Yards,Manhattan,West Chelsea/hudson Yards,CASH,0.4,7.02,7.20,0.00,12.20
2023-12-31,Manhattan,East Chelsea,Manhattan,Kips Bay,CREDIT_CARD,1.44,9.65,10.70,3.14,18.84
2023-12-31,Manhattan,Union Sq,Manhattan,Upper East Side South,CREDIT_CARD,3.14,15.33,17.00,6.60,28.60
2023-12-31,Manhattan,Sutton Place/turtle Bay North,Manhattan,Washington Heights South,CREDIT_CARD,7.7,18.75,33.10,7.62,45.72
2023-12-31,Manhattan,Midtown North,Manhattan,Upper East Side South,CREDIT_CARD,0.97,3.72,6.50,2.00,13.50
2023-12-31,Manhattan,Upper East Side North,Manhattan,Lincoln Square East,CREDIT_CARD,2.38,15.33,15.60,1.00,21.60
2023-12-31,Manhattan,Little Italy/nolita,Manhattan,Soho,CREDIT_CARD,0.53,4.55,5.80,2.16,12.96


### Step 26: SQL built-in function demonstration

In [0]:
%sql
SELECT
    trip_id,
    UPPER(pickup_borough) AS pickup_borough_upper,
    INITCAP(pickup_zone) AS pickup_zone_title,
    CONCAT_WS(' -> ', pickup_zone, dropoff_zone) AS route,
    LENGTH(pickup_zone) AS pickup_zone_length,
    ROUND(trip_distance_miles, 2) AS distance_miles,
    CEIL(trip_distance_miles) AS distance_ceiling,
    FLOOR(trip_distance_miles) AS distance_floor,
    ABS(total_amount) AS absolute_total,
    YEAR(pickup_timestamp) AS pickup_year,
    MONTH(pickup_timestamp) AS pickup_month,
    DAY(pickup_timestamp) AS pickup_day,
    HOUR(pickup_timestamp) AS pickup_hour,
    DATE_FORMAT(pickup_timestamp, 'EEEE') AS pickup_day_name,
    COALESCE(passenger_count, 0) AS passenger_count_clean,
    CASE
        WHEN trip_distance_miles < 2 THEN 'SHORT'
        WHEN trip_distance_miles < 8 THEN 'MEDIUM'
        ELSE 'LONG'
    END AS distance_category
FROM silver.enriched_taxi_trips
LIMIT 20;

trip_id,pickup_borough_upper,pickup_zone_title,route,pickup_zone_length,distance_miles,distance_ceiling,distance_floor,absolute_total,pickup_year,pickup_month,pickup_day,pickup_hour,pickup_day_name,passenger_count_clean,distance_category
004273c47c8f818a71a4d434ddee6598107fdee1ef9610b44ba30d4aff467565,MANHATTAN,Murray Hill,Murray Hill -> Lincoln Square East,11,1.98,2,1,21.84,2024,1,2,12,Tuesday,1,SHORT
00480d8056541509b71817923bb0b480cf5338010e36f69462824ec643c0948a,MANHATTAN,Alphabet City,Alphabet City -> Highbridge Park,13,11.8,12,11,30.00,2024,1,2,8,Tuesday,1,LONG
00687bd5fd7246ea683adce47f37caf6700c2cf16098cee9692dd19f49ea2e44,MANHATTAN,Gramercy,Gramercy -> Washington Heights North,8,10.08,11,10,56.28,2024,1,2,11,Tuesday,1,LONG
007abc802e784f00af8822bf5687501c246d2961618b1f85c7154edb6325ac94,MANHATTAN,Greenwich Village South,Greenwich Village South -> Times Sq/theatre District,23,2.2,3,2,18.80,2024,1,2,10,Tuesday,1,MEDIUM
008c441411ff182a120eb4000c4c2d843a34ce398d7f4bd251e1cabf9a718f11,MANHATTAN,Upper West Side North,Upper West Side North -> Yorkville West,21,1.4,2,1,15.95,2024,1,2,11,Tuesday,1,SHORT
0099527b329eb960ad35ad77d4e5da437fbf09a37b1a688d9418a2efd9c20ed4,QUEENS,Jfk Airport,Jfk Airport -> Times Sq/theatre District,11,23.6,24,23,82.69,2024,1,2,12,Tuesday,2,LONG
00a66f7df5ad6f27363e9716fe8c31f09ed15acd3e4b0282533f9b4363be849d,QUEENS,Maspeth,Maspeth -> Maspeth,7,8.02,9,8,94.69,2024,1,2,10,Tuesday,1,LONG
00aaff3f67c65bb794c35fe59883b556191e9face86558f45aa9fa8c2aeffacb,MANHATTAN,Upper East Side South,Upper East Side South -> Lincoln Square East,21,1.26,2,1,14.04,2024,1,2,13,Tuesday,1,SHORT
00aded6e5cf4554deae84f0a9c4285a6fbb90658b5ccd13563ab07627419e333,MANHATTAN,Midtown Center,Midtown Center -> Midtown North,14,0.78,1,0,14.28,2024,1,2,12,Tuesday,3,SHORT
00b58c15c3923dcee187113ad4b5f6361743f76614c0a73a874b5a86c5062ce4,MANHATTAN,Midtown North,Midtown North -> Midtown South,13,0.92,1,0,19.32,2024,1,2,12,Tuesday,2,SHORT


# Aggregations and actions
### Step 27: Aggregate by payment method

In [0]:
payment_summary_df = (
    spark.table("silver.enriched_taxi_trips")
    .groupBy("payment_method")
    .agg(
        F.count("*").alias("trip_count"),
        F.sum("total_amount").alias("total_revenue"),
        F.avg("total_amount").alias("average_trip_value"),
        F.min("total_amount").alias("minimum_trip_value"),
        F.max("total_amount").alias("maximum_trip_value"),
        F.stddev("total_amount").alias("revenue_standard_deviation"),
        F.approx_count_distinct("pickup_location_id").alias("approx_pickup_zone_count")
    )
    .orderBy(F.col("total_revenue").desc())
)

payment_summary_df.show(truncate=False)

+--------------+----------+-------------+------------------+------------------+------------------+--------------------------+------------------------+
|payment_method|trip_count|total_revenue|average_trip_value|minimum_trip_value|maximum_trip_value|revenue_standard_deviation|approx_pickup_zone_count|
+--------------+----------+-------------+------------------+------------------+------------------+--------------------------+------------------------+
|CREDIT_CARD   |74550     |2387513.80   |32.025671         |5.90              |486.10            |26.607177767359907        |213                     |
|CASH          |20408     |533469.22    |26.140201         |4.00              |1617.50           |25.669009337856018        |155                     |
|DISPUTE       |951       |25122.84     |26.417287         |0.00              |426.54            |27.333590570474783        |79                      |
|NO_CHARGE     |417       |10241.25     |24.559353         |0.00              |289.93         

### Step 28: Set operations

In [0]:
credit_card_df = spark.table("silver.enriched_taxi_trips").filter(
    F.col("payment_method") == "CREDIT_CARD"
)

cash_df = spark.table("silver.enriched_taxi_trips").filter(
    F.col("payment_method") == "CASH"
)

combined_df = credit_card_df.unionByName(cash_df)
print("Credit-card and cash trips:", combined_df.count())

Credit-card and cash trips: 94958


In [0]:
# Distinct pickup boroughs
spark.table("silver.enriched_taxi_trips").select("pickup_borough").distinct().show()

# Drop duplicate routes
spark.table("silver.enriched_taxi_trips").dropDuplicates(
    ["pickup_location_id", "dropoff_location_id"]
).select("pickup_location_id", "dropoff_location_id").show(10)

+--------------+
|pickup_borough|
+--------------+
| Staten Island|
|       Unknown|
|         Bronx|
|      Brooklyn|
|          NULL|
|           Ewr|
|     Manhattan|
|        Queens|
+--------------+

+------------------+-------------------+
|pickup_location_id|dropoff_location_id|
+------------------+-------------------+
|               148|                144|
|               164|                262|
|                87|                100|
|               239|                142|
|               132|                107|
|               229|                141|
|               141|                 48|
|               224|                 79|
|               100|                132|
|               100|                161|
+------------------+-------------------+
only showing top 10 rows


### Step 29: Common actions

In [0]:
action_df = spark.table("silver.enriched_taxi_trips")

print("count():", action_df.count())
print("first():", action_df.first())
print("take(3):", action_df.take(3))
print("head(2):", action_df.head(2))

# Use collect only after reducing the result to a small number of rows.
small_result = (
    action_df
    .groupBy("payment_method")
    .count()
    .orderBy("payment_method")
    .collect()
)

for row in small_result:
    print(row)

count(): 96326
first(): Row(vendor_id=2, pickup_timestamp=datetime.datetime(2002, 12, 31, 22, 59, 39), dropoff_timestamp=datetime.datetime(2002, 12, 31, 23, 5, 41), passenger_count=1, trip_distance_miles=0.63, rate_code_id=1, store_and_forward_flag='N', pickup_location_id=170, dropoff_location_id=170, payment_type_id=3, fare_amount=Decimal('6.50'), extra_amount=Decimal('0.00'), mta_tax=Decimal('0.50'), tip_amount=Decimal('0.00'), tolls_amount=Decimal('0.00'), improvement_surcharge=Decimal('1.00'), total_amount=Decimal('10.50'), congestion_surcharge=Decimal('2.50'), source_file='Yello_tripdata_2024_01.parquet', ingestion_timestamp=datetime.datetime(2026, 7, 23, 15, 18, 21, 935928), trip_duration_minutes=6.03, pickup_date=datetime.date(2002, 12, 31), pickup_year=2002, pickup_month=12, pickup_day=31, pickup_hour=22, pickup_day_name='Tuesday', payment_method='NO_CHARGE', distance_band='SHORT', tip_percentage=0.0, trip_id='aeaffb60bff6e16834ddecac71c4e2c4fb70f56de0048584e27e2e47d5c8496d', p

### Step 30: Rank pickup zones by revenue within each borough

In [0]:
zone_revenue_df = (
    spark.table("silver.enriched_taxi_trips")
    .groupBy("pickup_borough", "pickup_zone")
    .agg(
        F.count("*").alias("trip_count"),
        F.sum("total_amount").alias("total_revenue")
    )
)

borough_rank_window = Window.partitionBy("pickup_borough").orderBy(
    F.col("total_revenue").desc()
)

ranked_zone_df = (
    zone_revenue_df
    .withColumn("row_number", F.row_number().over(borough_rank_window))
    .withColumn("rank", F.rank().over(borough_rank_window))
    .withColumn("dense_rank", F.dense_rank().over(borough_rank_window))
)

ranked_zone_df.filter(F.col("dense_rank") <= 5).show(50, truncate=False)

+--------------+---------------------------------+----------+-------------+----------+----+----------+
|pickup_borough|pickup_zone                      |trip_count|total_revenue|row_number|rank|dense_rank|
+--------------+---------------------------------+----------+-------------+----------+----+----------+
|NULL          |Outside Of Nyc                   |21        |1709.07      |1         |1   |1         |
|Bronx         |Mott Haven/port Morris           |13        |1909.94      |1         |1   |1         |
|Bronx         |Co-op City                       |10        |468.36       |2         |2   |2         |
|Bronx         |Van Cortlandt Village            |9         |377.45       |3         |3   |3         |
|Bronx         |East Concourse/concourse Village |12        |317.34       |4         |4   |4         |
|Bronx         |University Heights/morris Heights|7         |297.50       |5         |5   |5         |
|Brooklyn      |Downtown Brooklyn/metrotech      |81        |2717.56     

Step 31: Lag, lead, and running total

In [0]:
daily_revenue_df = (
    spark.table("silver.enriched_taxi_trips")
    .groupBy("pickup_date")
    .agg(F.sum("total_amount").alias("daily_revenue"))
)

date_window = Window.orderBy("pickup_date")
running_window = (
    Window.orderBy("pickup_date")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

daily_trend_df = (
    daily_revenue_df
    .withColumn("previous_day_revenue", F.lag("daily_revenue", 1).over(date_window))
    .withColumn("next_day_revenue", F.lead("daily_revenue", 1).over(date_window))
    .withColumn("running_revenue", F.sum("daily_revenue").over(running_window))
    .withColumn(
        "revenue_change",
        F.col("daily_revenue") - F.col("previous_day_revenue")
    )
)

daily_trend_df.show(40, truncate=False)

+-----------+-------------+--------------------+----------------+---------------+--------------+
|pickup_date|daily_revenue|previous_day_revenue|next_day_revenue|running_revenue|revenue_change|
+-----------+-------------+--------------------+----------------+---------------+--------------+
|2002-12-31 |10.50        |NULL                |9.40            |10.50          |NULL          |
|2009-01-01 |9.40         |10.50               |224.62          |19.90          |-1.10         |
|2023-12-31 |224.62       |9.40                |2108801.72      |244.52         |215.22        |
|2024-01-01 |2108801.72   |224.62              |847282.87       |2109046.24     |2108577.10    |
|2024-01-02 |847282.87    |2108801.72          |18.00           |2956329.11     |-1261518.85   |
|2024-01-03 |18.00        |847282.87           |NULL            |2956347.11     |-847264.87    |
+-----------+-------------+--------------------+----------------+---------------+--------------+



# Gold layer
### Step 32: Create the Gold zone dimension

In [0]:
%sql
CREATE OR REPLACE TABLE gold.dim_taxi_zone
USING DELTA
AS
SELECT
  location_id as zone_key,
  borough,
  zone_name,
  service_zone,
  CONCAT_WS('  - ',borough,zone_name) AS full_zone_name
FROM
  silver.clean_taxi_zones

num_affected_rows,num_inserted_rows


### Step 33: Create the Gold trip fact table

In [0]:
%sql
CREATE OR REPLACE TABLE gold.fact_taxi_trip
USING DELTA
AS
SELECT
    trip_id,
    pickup_date,
    pickup_timestamp,
    dropoff_timestamp,
    pickup_location_id,
    dropoff_location_id,
    vendor_id,
    payment_type_id,
    payment_method,
    passenger_count,
    trip_distance_miles,
    trip_duration_minutes,
    fare_amount,
    extra_amount,
    mta_tax,
    tip_amount,
    tolls_amount,
    improvement_surcharge,
    congestion_surcharge,
    total_amount,
    tip_percentage,
    distance_band
FROM silver.enriched_taxi_trips;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT count(*) FROM gold.fact_taxi_trip

count(*)
96326


### Step 34: Create daily KPI table

In [0]:
%sql
CREATE OR REPLACE TABLE gold.daily_taxi_kpis
USING DELTA
AS
SELECT
    pickup_date,
    COUNT(*) AS total_trips,
    SUM(passenger_count) AS total_passengers,
    ROUND(SUM(total_amount), 2) AS total_revenue,
    ROUND(AVG(total_amount), 2) AS average_trip_value,
    ROUND(AVG(trip_distance_miles), 2) AS average_trip_distance,
    ROUND(AVG(trip_duration_minutes), 2) AS average_trip_duration_minutes,
    ROUND(SUM(tip_amount), 2) AS total_tip_amount,
    ROUND(
        100.0 * SUM(CASE WHEN payment_method = 'CREDIT_CARD' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS credit_card_trip_percentage
FROM silver.enriched_taxi_trips
GROUP BY pickup_date;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM gold.daily_taxi_kpis

pickup_date,total_trips,total_passengers,total_revenue,average_trip_value,average_trip_distance,average_trip_duration_minutes,total_tip_amount,credit_card_trip_percentage
2024-01-01,67779,105166,2108801.72,31.11,4.33,16.86,252023.67,77.75
2024-01-02,28534,39109,847282.87,29.69,4.25,16.57,96069.66,76.55
2023-12-31,10,19,224.62,22.46,2.6,10.16,26.27,80.00
2024-01-03,1,6,18.00,18.00,1.41,6.98,3.00,100.00
2002-12-31,1,1,10.50,10.50,0.63,6.03,0.00,0.00
2009-01-01,1,1,9.40,9.40,0.46,3.0,0.00,0.00


### Step 36: Create payment-method summary

In [0]:
%sql
CREATE OR REPLACE TABLE gold.payment_method_summary
USING DELTA
AS
SELECT
    payment_method,
    COUNT(*) AS trip_count,
    ROUND(SUM(total_amount), 2) AS total_revenue,
    ROUND(AVG(total_amount), 2) AS average_trip_value,
    ROUND(SUM(tip_amount), 2) AS total_tip_amount,
    ROUND(AVG(tip_percentage), 2) AS average_tip_percentage
FROM silver.enriched_taxi_trips
GROUP BY payment_method;


num_affected_rows,num_inserted_rows


### Step 37: Create top routes with a CTE and ranking

In [0]:
%sql
CREATE OR REPLACE TABLE gold.top_taxi_routes
USING DELTA
AS
WITH route_summary AS (
    SELECT
        pickup_borough,
        pickup_zone,
        dropoff_borough,
        dropoff_zone,
        COUNT(*) AS trip_count,
        ROUND(SUM(total_amount), 2) AS total_revenue,
        ROUND(AVG(trip_distance_miles), 2) AS average_distance_miles,
        ROUND(AVG(trip_duration_minutes), 2) AS average_duration_minutes
    FROM silver.enriched_taxi_trips
    WHERE pickup_zone IS NOT NULL
      AND dropoff_zone IS NOT NULL
    GROUP BY
        pickup_borough,
        pickup_zone,
        dropoff_borough,
        dropoff_zone
),
ranked_routes AS (
    SELECT
        *,
        DENSE_RANK() OVER (
            PARTITION BY pickup_borough
            ORDER BY trip_count DESC
        ) AS route_rank_in_pickup_borough
    FROM route_summary
)
SELECT *
FROM ranked_routes
WHERE route_rank_in_pickup_borough <= 10;

num_affected_rows,num_inserted_rows


### Query 1: Daily revenue trend

In [0]:
%sql
SELECT *
FROM gold.daily_taxi_kpis
ORDER BY pickup_date;

pickup_date,total_trips,total_passengers,total_revenue,average_trip_value,average_trip_distance,average_trip_duration_minutes,total_tip_amount,credit_card_trip_percentage
2002-12-31,1,1,10.50,10.50,0.63,6.03,0.00,0.00
2009-01-01,1,1,9.40,9.40,0.46,3.0,0.00,0.00
2023-12-31,10,19,224.62,22.46,2.6,10.16,26.27,80.00
2024-01-01,67779,105166,2108801.72,31.11,4.33,16.86,252023.67,77.75
2024-01-02,28534,39109,847282.87,29.69,4.25,16.57,96069.66,76.55
2024-01-03,1,6,18.00,18.00,1.41,6.98,3.00,100.00


# Data-quality validation
### Validate Bronze, Silver, and Gold counts

In [0]:
%sql
SELECT 'bronze_trips' AS table_name, COUNT(*) AS row_count
FROM bronze.nyc_yellow_taxi_trips_raw

UNION ALL

SELECT 'silver_clean_trips', COUNT(*)
FROM silver.clean_taxi_trips

UNION ALL

SELECT 'silver_invalid_trips', COUNT(*)
FROM silver.invalid_taxi_trips

UNION ALL

SELECT 'gold_fact_trips', COUNT(*)
FROM gold.fact_taxi_trip;

table_name,row_count
bronze_trips,100000
silver_clean_trips,96326
silver_invalid_trips,3674
gold_fact_trips,96326


### Step 39: Confirm no duplicate trip IDs remain

In [0]:
%sql
SELECT trip_id, COUNT(*) AS duplicate_count
FROM silver.clean_taxi_trips
GROUP BY trip_id
HAVING COUNT(*) > 1;

trip_id,duplicate_count


###  Execution plan, partitions, and caching
###  View the execution plan

In [0]:
analysis_df = (
    spark.table("silver.enriched_taxi_trips")
    .filter(F.col("pickup_borough") == "Manhattan")
    .groupBy("pickup_zone")
    .agg(F.sum("total_amount").alias("revenue"))
    .orderBy(F.col("revenue").desc())
)

analysis_df.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (14)
+- == Initial Plan ==
   PhotonResultStage (13)
   +- PhotonColumnarToRow (12)
      +- PhotonSort (11)
         +- PhotonShuffleExchangeSource (10)
            +- PhotonShuffleMapStage (9)
               +- PhotonShuffleExchangeSink (8)
                  +- PhotonGroupingAgg (7)
                     +- PhotonShuffleExchangeSource (6)
                        +- PhotonShuffleMapStage (5)
                           +- PhotonShuffleExchangeSink (4)
                              +- PhotonGroupingAgg (3)
                                 +- PhotonProject (2)
                                    +- PhotonScan parquet workspace.silver.enriched_taxi_trips (1)


(1) PhotonScan parquet workspace.silver.enriched_taxi_trips
Output [4]: [total_amount#30751, pickup_borough#30766, pickup_zone#30767, pickup_date#30756]
DictionaryFilters: [(pickup_borough#30766 = Manhattan)]
Location: PreparedDeltaFileIndex [s3://dbstorage-prod-ixuxc19cqn/uc/17707bfb-4ea0-4893-8

### Inspect and control partitions

In [0]:
partition_demo_df = spark.table("silver.enriched_taxi_trips")

print("Current partitions:", partition_demo_df.rdd.getNumPartitions())

more_partitions_df = partition_demo_df.repartition(8, "pickup_borough")
fewer_partitions_df = more_partitions_df.coalesce(4)

print("After repartition:", more_partitions_df.rdd.getNumPartitions())
print("After coalesce:", fewer_partitions_df.rdd.getNumPartitions())

---------------------------------------------------------------------------
PySparkNotImplementedError                Traceback (most recent call last)
File <command-8940903305098314>, line 3
      1 partition_demo_df = spark.table("silver.enriched_taxi_trips")
----> 3 print("Current partitions:", partition_demo_df.rdd.getNumPartitions())
      5 more_partitions_df = partition_demo_df.repartition(8, "pickup_borough")
      6 fewer_partitions_df = more_partitions_df.coalesce(4)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:2373, in DataFrame.rdd(self)
   2371 @property
   2372 def rdd(self) -> "RDD[Row]":
-> 2373     raise PySparkNotImplementedError(
   2374         errorClass="NOT_IMPLEMENTED",
   2375         messageParameters={"feature": "rdd"},
   2376     )

PySparkNotImplementedError: [NOT_IMPLEMENTED] Using custom code using PySpark RDDs is not allowed on serverless compute. We suggest using mapInPandas or mapInArrow for the most common us